# Tugas UTS: Linear Regression & Logistic Regression
**Mata Kuliah:** Machine Learning

**Nama:** Muhammad Raka Fadillah  
**NIM:** 231011402919  
**Kelas:** 05TPLE005

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NIM = 231011402919
RANDOM_STATE = NIM % (2**32 - 1)

np.set_printoptions(precision=4, suppress=True)

print("Nama: Muhammad Raka Fadillah")
print("NIM:", NIM)
print("Kelas: 05TPLE005")
print("Random State:", RANDOM_STATE)
print("Semua library berhasil diimpor!")

Nama: Muhammad Raka Fadillah
NIM: 231011402919
Kelas: 05TPLE005
Random State: 3378136284
Semua library berhasil diimpor!


## Fungsi dasar yang diimplementasikan sendiri dengan NumPy

In [31]:
def train_test_split(X, y, test_size=0.2, random_state=42):
    X = np.asarray(X); y = np.asarray(y)
    rng = np.random.default_rng(random_state)
    indices = rng.permutation(len(X))
    n_test = int(np.ceil(len(X) * test_size))
    test_idx, train_idx = indices[:n_test], indices[n_test:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def confusion_matrix(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int); y_pred = np.asarray(y_pred).astype(int)
    return np.array([[np.sum((y_true==0)&(y_pred==0)), np.sum((y_true==0)&(y_pred==1))],
                     [np.sum((y_true==1)&(y_pred==0)), np.sum((y_true==1)&(y_pred==1))]])

def accuracy_score(y_true, y_pred):
    return float(np.mean(np.asarray(y_true)==np.asarray(y_pred)))

def classification_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    return {'Accuracy': accuracy_score(y_true,y_pred), 'Precision': precision, 'Recall': recall, 'F1-score': f1}, cm

class LinearRegression:
    def fit(self, X, y):
        X = np.asarray(X).reshape(-1,1); y = np.asarray(y).reshape(-1,1)
        Xb = np.c_[np.ones((len(X),1)), X]
        self.theta = np.linalg.pinv(Xb.T @ Xb) @ Xb.T @ y
        self.intercept_ = float(self.theta[0,0]); self.coef_ = float(self.theta[1,0])
        return self
    def predict(self, X):
        X = np.asarray(X).reshape(-1,1)
        return self.intercept_ + self.coef_*X[:,0]

def regression_metrics(y_true, y_pred):
    y_true=np.asarray(y_true).ravel(); y_pred=np.asarray(y_pred).ravel()
    err=y_true-y_pred; mse=np.mean(err**2)
    denom=np.sum((y_true-np.mean(y_true))**2)
    return {'MAE':np.mean(np.abs(err)), 'MSE':mse, 'RMSE':np.sqrt(mse), 'R2':1-np.sum(err**2)/denom if denom else np.nan}

class LinearRegressionGD:
    def __init__(self, learning_rate=0.01, n_iterations=5000):
        self.learning_rate=learning_rate; self.n_iterations=n_iterations
    def fit(self, X, y):
        X=np.asarray(X).reshape(-1,1); y=np.asarray(y).ravel()
        self.mu=X.mean(axis=0); self.sd=X.std(axis=0); self.sd[self.sd==0]=1
        Z=(X-self.mu)/self.sd; Zb=np.c_[np.ones(len(Z)),Z]
        theta=np.zeros(2); self.loss_history=[]
        for _ in range(self.n_iterations):
            pred=Zb@theta; grad=(2/len(y))*Zb.T@(pred-y)
            theta -= self.learning_rate*grad
            with np.errstate(over='ignore', invalid='ignore'):
                loss=np.mean((Zb@theta-y)**2)
            self.loss_history.append(loss)
            if not np.isfinite(loss) or not np.all(np.isfinite(theta)):
                break
        self.intercept_=float(theta[0] - np.sum((theta[1:]*self.mu)/self.sd))
        self.coef_=float(theta[1]/self.sd[0])
        return self
    def predict(self,X): return self.intercept_+self.coef_*np.asarray(X).reshape(-1)

class LogisticRegression:
    def __init__(self, learning_rate=0.1, n_iterations=10000):
        self.learning_rate=learning_rate; self.n_iterations=n_iterations
    def fit(self,X,y):
        X=np.asarray(X,dtype=float); y=np.asarray(y,dtype=float).ravel()
        self.mu=X.mean(axis=0); self.sd=X.std(axis=0); self.sd[self.sd==0]=1
        Z=(X-self.mu)/self.sd; Zb=np.c_[np.ones(len(Z)),Z]
        theta=np.zeros(Zb.shape[1]); self.loss_history=[]
        for _ in range(self.n_iterations):
            p=sigmoid(Zb@theta)
            grad=(Zb.T@(p-y))/len(y)
            theta -= self.learning_rate*grad
            p2=np.clip(sigmoid(Zb@theta),1e-12,1-1e-12)
            loss=-np.mean(y*np.log(p2)+(1-y)*np.log(1-p2))
            self.loss_history.append(loss)
        self.coef_=theta[1:]/self.sd
        self.intercept_=float(theta[0]-np.sum(theta[1:]*self.mu/self.sd))
        return self
    def predict_proba(self,X):
        X=np.asarray(X,dtype=float); return sigmoid(self.intercept_+X@self.coef_)
    def predict(self,X,threshold=0.5): return (self.predict_proba(X)>=threshold).astype(int)


---
# Bagian A – Linear Regression (35%)

### A1 – Load & Visualisasi Data

In [32]:
ads = pd.read_csv('Ads_Data.csv')
print('Lima data pertama:'); display(ads.head())
print('Jumlah baris:', len(ads))
X_ads = ads[['BiayaIklan']].to_numpy()
y_ads = ads['Penjualan'].to_numpy()
print('Shape X:', X_ads.shape, '| shape y:', y_ads.shape)
plt.figure(figsize=(7,4)); plt.scatter(X_ads[:,0], y_ads); plt.title('Biaya Iklan vs Penjualan')
plt.xlabel('Biaya Iklan (juta Rp)'); plt.ylabel('Penjualan (juta Rp)'); plt.grid(alpha=.25); plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'Ads_Data.csv'

### A2 – Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_ads, y_ads, test_size=0.2, random_state=RANDOM_STATE)
print('X_train:',X_train.shape,'X_test:',X_test.shape,'y_train:',y_train.shape,'y_test:',y_test.shape)

NameError: name 'X_ads' is not defined

### A3 – Training dengan Normal Equation

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
print(f'Intercept (b): {lr.intercept_:.4f}')
print(f'Koefisien (m): {lr.coef_:.4f}')
print(f'Persamaan: y_hat = {lr.intercept_:.4f} + ({lr.coef_:.4f} × BiayaIklan)')

NameError: name 'X_train' is not defined

**Jawaban A3 (interpretasi):** Intercept adalah prediksi penjualan ketika biaya iklan bernilai nol, sehingga interpretasinya perlu hati-hati jika nol berada di luar rentang pengamatan. Koefisien menunjukkan perubahan rata-rata penjualan (juta rupiah) untuk setiap kenaikan biaya iklan sebesar 1 juta rupiah; lihat nilai koefisien yang dicetak di atas untuk menentukan besar dan arahnya.

### A4 – Evaluasi Model

In [ ]:
pred_train = lr.predict(X_train); pred_test = lr.predict(X_test)
metrics_table = pd.DataFrame([regression_metrics(y_train,pred_train), regression_metrics(y_test,pred_test)], index=['Training','Testing'])
display(metrics_table.round(4))

NameError: name 'lr' is not defined

**Jawaban A4 (analisis):** MAE adalah rata-rata besar kesalahan absolut, sedangkan RMSE memberi penalti lebih besar pada kesalahan yang besar dan tetap memakai satuan juta rupiah. R² mendekati 1 menunjukkan model menjelaskan variasi data dengan baik; bandingkan R² dan RMSE training dengan testing di tabel. Jika hasil testing jauh lebih buruk, kemampuan generalisasi model perlu dipertanyakan.

### A5 – Gradient Descent & Learning Rate

In [ ]:
rates=[0.001,0.02,0.1]
gd_models={}
plt.figure(figsize=(8,5))
for rate in rates:
    model=LinearRegressionGD(learning_rate=rate,n_iterations=5000).fit(X_train,y_train)
    gd_models[rate]=model
    print(f'lr={rate}: intercept={model.intercept_:.4f}, koefisien={model.coef_:.4f}, iterasi={len(model.loss_history)}, loss akhir={model.loss_history[-1]}')
    losses=np.asarray(model.loss_history,dtype=float)
    plt.plot(np.arange(1,len(losses)+1),np.maximum(losses,1e-12),label=f'lr={rate}')
plt.yscale('log'); plt.xlabel('Iterasi'); plt.ylabel('MSE Loss (log scale)'); plt.title('Gradient Descent: Loss vs Iterasi'); plt.legend(); plt.grid(alpha=.25); plt.show()
print(f'Normal equation: intercept={lr.intercept_:.4f}, koefisien={lr.coef_:.4f}')

NameError: name 'X_train' is not defined

<Figure size 800x500 with 0 Axes>

**Jawaban A5 (analisis):** Bandingkan kurva dan parameter akhir tiap learning rate dengan normal equation. Learning rate kecil biasanya turun lebih lambat, nilai menengah dapat mencapai solusi lebih cepat, sedangkan learning rate terlalu besar dapat membuat loss berosilasi atau divergen. Pada implementasi ini fitur distandardisasi agar eksperimen lebih stabil; jika kurva tetap tidak stabil atau menghasilkan nilai tidak hingga, itu menandakan langkah pembaruan terlalu besar.

### A6 – Visualisasi & Prediksi

In [ ]:
xline=np.linspace(X_test.min(),X_test.max(),200)
plt.figure(figsize=(7,4)); plt.scatter(X_test[:,0],y_test,label='Data testing'); plt.plot(xline,lr.predict(xline.reshape(-1,1)),color='red',label='Garis regresi')
plt.title('Data Testing dan Garis Regresi'); plt.xlabel('Biaya Iklan (juta Rp)'); plt.ylabel('Penjualan (juta Rp)'); plt.legend(); plt.grid(alpha=.25); plt.show()
biaya_baru=np.array([4.5,7.0,12.0]).reshape(-1,1)
for x,p in zip(biaya_baru[:,0],lr.predict(biaya_baru)):
    print(f'Biaya iklan {x:g} juta -> prediksi penjualan {p:.2f} juta Rp')

NameError: name 'X_test' is not defined

**Jawaban A6 (analisis):** Prediksi 4,5 dan 7 juta lebih dapat dipertimbangkan jika nilainya masih berada di dalam rentang biaya iklan yang diamati. Prediksi untuk 12 juta merupakan ekstrapolasi karena berada di luar rentang data; hasilnya lebih tidak pasti sebab pola hubungan di luar pengamatan belum teruji dan dapat berubah.

---
# Bagian B – Logistic Regression (40%)

### B1 – Load & Visualisasi Data

In [ ]:
emails=pd.read_csv('Email_Data.csv')
print('Lima data pertama:'); display(emails.head())
print('Jumlah label:'); display(emails['Spam'].map({0:'Bukan Spam',1:'Spam'}).value_counts().to_frame('Jumlah'))
X_email=emails[['JumlahLink','KataPromo']].to_numpy(dtype=float); y_email=emails['Spam'].to_numpy(dtype=int)
plt.figure(figsize=(7,5))
for label,name in [(0,'Bukan Spam'),(1,'Spam')]:
    mask=y_email==label; plt.scatter(X_email[mask,0],X_email[mask,1],label=name,alpha=.8)
plt.xlabel('Jumlah Link'); plt.ylabel('Jumlah Kata Promo'); plt.title('Sebaran Email Spam dan Bukan Spam'); plt.legend(); plt.grid(alpha=.2); plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'Email_Data.csv'

### B2 – Split Data & Training

In [ ]:
Xe_train,Xe_test,ye_train,ye_test=train_test_split(X_email,y_email,test_size=0.25,random_state=RANDOM_STATE)
logreg=LogisticRegression(learning_rate=0.1,n_iterations=10000).fit(Xe_train,ye_train)
print('Ukuran data:',Xe_train.shape,Xe_test.shape)
print(f'Intercept b: {logreg.intercept_:.6f}')
print('Koefisien:',logreg.coef_)
print(f'P(Spam)=sigmoid({logreg.intercept_:.4f} + ({logreg.coef_[0]:.4f} × JumlahLink) + ({logreg.coef_[1]:.4f} × KataPromo))')
plt.figure(figsize=(7,4)); plt.plot(logreg.loss_history); plt.xlabel('Iterasi'); plt.ylabel('Log Loss'); plt.title('Logistic Regression: Log Loss vs Iterasi'); plt.grid(alpha=.25); plt.show()
print('Log loss awal:',round(logreg.loss_history[0],5),'| akhir:',round(logreg.loss_history[-1],5))

NameError: name 'X_email' is not defined

**Jawaban B2 (analisis):** Jika log loss menurun lalu relatif stabil, model sedang belajar dan pembaruannya mulai konvergen. Koefisien positif meningkatkan probabilitas spam ketika fitur terkait bertambah, sedangkan koefisien negatif menurunkannya; interpretasikan tanda koefisien hasil aktual di atas.

### B3 – Evaluasi Model

In [ ]:
prob_test=logreg.predict_proba(Xe_test); pred_test_email=(prob_test>=0.5).astype(int)
class_metrics,cm=classification_metrics(ye_test,pred_test_email)
print('Confusion matrix [[TN, FP], [FN, TP]]:'); print(cm)
display(pd.DataFrame([class_metrics]).round(4))
tn,fp,fn,tp=cm.ravel(); print(f'TN={tn}, FP={fp}, FN={fn}, TP={tp}')

NameError: name 'logreg' is not defined

**Jawaban B3 (analisis):** FP adalah email bukan spam yang salah ditandai sebagai spam sehingga pesan penting bisa masuk folder spam. FN adalah email spam yang lolos ke kotak masuk. Untuk pengguna yang khawatir kehilangan pesan penting, FP dapat lebih merugikan; tetapi pada sistem dengan ancaman phishing tinggi, FN juga berisiko. Pilihan bergantung pada prioritas keamanan dan kenyamanan.

### B4 – Decision Boundary

In [ ]:
m1,m2=logreg.coef_
if abs(m2)>1e-12:
    print(f'Decision boundary: KataPromo = -({logreg.intercept_:.4f} + ({m1:.4f} × JumlahLink)) / ({m2:.4f})')
    x_min,x_max=Xe_test[:,0].min()-1,Xe_test[:,0].max()+1
    y_min,y_max=Xe_test[:,1].min()-1,Xe_test[:,1].max()+1
    xx,yy=np.meshgrid(np.linspace(x_min,x_max,200),np.linspace(y_min,y_max,200))
    grid=np.c_[xx.ravel(),yy.ravel()]; zz=logreg.predict_proba(grid).reshape(xx.shape)
    plt.figure(figsize=(8,6)); plt.contourf(xx,yy,zz,levels=np.linspace(0,1,11),alpha=.35,cmap='RdYlBu'); plt.colorbar(label='Probabilitas Spam')
    wrong=pred_test_email!=ye_test
    for label,name,marker in [(0,'Bukan Spam','o'),(1,'Spam','^')]:
        mask=ye_test==label; plt.scatter(Xe_test[mask,0],Xe_test[mask,1],c=ye_test[mask],cmap='RdYlBu',edgecolors='black',marker=marker,label=f'Aktual {name}')
    if abs(m2)>1e-12:
        xvals=np.linspace(x_min,x_max,200); yvals=-(logreg.intercept_+m1*xvals)/m2; plt.plot(xvals,yvals,'k--',label='Decision boundary')
    if np.any(wrong): plt.scatter(Xe_test[wrong,0],Xe_test[wrong,1],s=180,facecolors='none',edgecolors='magenta',linewidths=2,label='Salah klasifikasi')
    plt.xlabel('Jumlah Link'); plt.ylabel('Jumlah Kata Promo'); plt.title('Decision Boundary dan Data Testing'); plt.legend(); plt.show()
    print('Data testing salah klasifikasi (fitur, aktual, prediksi):')
    for i in np.where(wrong)[0]: print(Xe_test[i],int(ye_test[i]),int(pred_test_email[i]))
else:
    print('Koefisien KataPromo mendekati nol; decision boundary tidak dapat ditulis sebagai fungsi KataPromo.')

NameError: name 'logreg' is not defined

**Jawaban B4 (analisis):** Garis decision boundary memisahkan area dengan probabilitas prediksi spam di bawah dan di atas 0,5. Titik yang dilingkari magenta adalah email testing yang prediksinya berbeda dari label aktual; jumlahnya menunjukkan kesalahan klasifikasi pada data uji.

### B5 – Eksperimen Threshold

In [ ]:
threshold_rows=[]
for threshold in [0.3,0.5,0.7]:
    p=(prob_test>=threshold).astype(int); met,cm_t=classification_metrics(ye_test,p)
    threshold_rows.append({'Threshold':threshold,'TN':cm_t[0,0],'FP':cm_t[0,1],'FN':cm_t[1,0],'TP':cm_t[1,1],**{k:met[k] for k in ['Precision','Recall','F1-score']}})
threshold_table=pd.DataFrame(threshold_rows); display(threshold_table.round(4))

NameError: name 'prob_test' is not defined

**Jawaban B5 (analisis):** Threshold lebih rendah cenderung mengklasifikasikan lebih banyak email sebagai spam, sehingga recall biasanya naik tetapi false positive juga dapat bertambah. Threshold lebih tinggi membuat model lebih selektif, sehingga precision dapat meningkat tetapi sebagian spam bisa terlewat. Pilih threshold dengan mempertimbangkan tabel hasil serta biaya relatif FP dan FN; untuk meminimalkan email penting yang salah masuk spam, threshold lebih tinggi dapat dipertimbangkan.

### B6 – Prediksi Email Baru

In [ ]:
new_emails=pd.DataFrame({'Email':['E1','E2','E3','E4'],'JumlahLink':[1,5,9,2],'KataPromo':[2,6,10,11]})
THRESHOLD_PILIHAN=0.5
new_X=new_emails[['JumlahLink','KataPromo']].to_numpy(dtype=float)
new_prob=logreg.predict_proba(new_X); new_pred=(new_prob>=THRESHOLD_PILIHAN).astype(int)
new_emails['Probabilitas Spam']=new_prob; new_emails['Prediksi']=np.where(new_pred==1,'Spam','Bukan Spam')
display(new_emails.round(4))

NameError: name 'logreg' is not defined

---
# Bagian C – Pemahaman Konsep (15%)

**Jawaban C1:** Logistic Regression termasuk klasifikasi karena menghasilkan probabilitas suatu contoh masuk ke kelas tertentu, lalu probabilitas itu diubah menjadi label seperti spam atau bukan spam menggunakan threshold. Fungsi sigmoid memetakan nilai linear apa pun menjadi angka antara 0 dan 1. Angka ini dapat ditafsirkan sebagai probabilitas kelas positif.

**Jawaban C2:** Normal equation menghitung parameter regresi secara langsung melalui operasi matriks dan tidak memerlukan iterasi. Gradient descent memperbarui parameter sedikit demi sedikit berdasarkan gradien loss. Gradient descent lebih berguna pada dataset besar atau saat penyelesaian langsung mahal secara komputasi. Kecepatan dan keberhasilannya dipengaruhi learning rate.

**Jawaban C3:** Data testing digunakan untuk mengukur kemampuan model pada data yang tidak dipakai saat pelatihan. Evaluasi hanya pada data training dapat memberi gambaran terlalu optimistis. Overfitting terjadi ketika model terlalu menyesuaikan diri dengan data training, termasuk noise, sehingga kinerjanya memburuk pada data baru.

**Jawaban C4:** Linear Regression memprediksi nilai kontinu sehingga MSE mengukur kuadrat selisih antara nilai aktual dan prediksi. Logistic Regression memprediksi probabilitas kelas, sehingga log loss mengukur seberapa baik probabilitas tersebut cocok dengan label sebenarnya. Log loss memberi penalti besar ketika model sangat yakin pada prediksi yang salah.

**Jawaban C5:** Hasil dapat berbeda karena pembagian training-testing dipengaruhi `random_state`, yang pada tugas ini ditentukan dari dua digit terakhir NIM. Hasil juga bisa dipengaruhi inisialisasi parameter, learning rate, jumlah iterasi, dan kriteria konvergensi. Karena data training yang digunakan bisa berbeda, koefisien dan metrik evaluasi juga dapat berubah.

---
# Bonus (+5) – Perbandingan dengan scikit-learn

In [ ]:
try:
    from sklearn.linear_model import LinearRegression as SkLinearRegression, LogisticRegression as SkLogisticRegression
    from sklearn.metrics import mean_squared_error, r2_score, accuracy_score as sk_accuracy
    sk_lr=SkLinearRegression().fit(X_train,y_train)
    print('LinearRegression sklearn intercept/coef:',sk_lr.intercept_,sk_lr.coef_)
    print('Implementasi sendiri intercept/coef:',lr.intercept_,lr.coef_)
    print('Sklearn test R2:',r2_score(y_test,sk_lr.predict(X_test)))
    sk_log=SkLogisticRegression(C=1e6,max_iter=10000).fit(Xe_train,ye_train)
    sk_pred=sk_log.predict(Xe_test)
    print('Logistic sklearn intercept/coef:',sk_log.intercept_,sk_log.coef_)
    print('Implementasi sendiri intercept/coef:',logreg.intercept_,logreg.coef_)
    print('Sklearn accuracy:',sk_accuracy(ye_test,sk_pred),'| NumPy accuracy:',accuracy_score(ye_test,pred_test_email))
except ImportError:
    print('scikit-learn tidak terpasang; lewati bagian bonus atau install hanya untuk bonus.')

**Jawaban Bonus (analisis):** Linear Regression umumnya menghasilkan parameter yang sangat dekat karena keduanya meminimalkan jumlah kuadrat residual. Logistic Regression dapat berbeda karena scikit-learn memakai regularisasi secara default; nilai `C` besar mengurangi pengaruh regularisasi, tetapi optimisasi numerik dan kriteria berhenti tetap dapat menimbulkan selisih.